# Feature Engineering — Partie 3

**Objectifs** : visualiser l’encodage cyclique, les features temporelles, le lissage bayésien des encodeurs cible, comparer bayésien vs cross-fitted, le benchmark d’imputation et une série temporelle agrégée (zone 132) + harmoniques de Fourier.


In [ ]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

ROOT = Path("..").resolve()

import logging

logging.getLogger().setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt

from src.optim.benchmarks import generate_taxi_df
from src.features.temporal import TemporalFeaturizer, encode_cyclic
from src.features.encoding import (
    BayesianTargetEncoder,
    CrossFittedTargetEncoder,
    compare_smoothing_effect,
)
from src.features.windowing import prepare_dataframe, build_activity_multiindex, top_zones_by_tip
from src.features.imputation import ImputationBenchmark
from src.features.timeseries import TimeSeriesFeaturizer, aggregate_hourly, add_fourier_features


def style_plotly(fig):
    fig.update_layout(title_font_size=16, template="plotly_dark")
    return fig


## Encodage cyclique sin/cos de l’heure

Le nuage \((\sin, \cos)\) forme un **cercle** : 0h et 23h sont proches dans l’espace des features.

In [ ]:
hours = pd.Series(range(24))
sin_h, cos_h = encode_cyclic(hours, 24)

fig = px.scatter(
    x=sin_h,
    y=cos_h,
    color=hours,
    color_continuous_scale="Turbo",
    title="Encodage Cyclique de l'Heure (sin/cos)",
    labels={"x": "sin", "y": "cos", "color": "Heure"},
)
for h, label in [(0, "0h"), (6, "6h"), (12, "12h"), (18, "18h")]:
    fig.add_annotation(x=float(sin_h.iloc[h]), y=float(cos_h.iloc[h]), text=label, showarrow=True, arrowhead=2)
style_plotly(fig)
fig.show()


## Distributions des features temporelles

`TemporalFeaturizer` sur 100k lignes ; sous-graphes des variables discrètes clés.

In [ ]:
df_raw = generate_taxi_df(100_000)
tf = TemporalFeaturizer(cyclic_encoding=False, add_holidays=False, add_rush_hours=True)
df_t = tf.run(df_raw.copy())

cols_plot = [
    "pickup_hour",
    "pickup_weekday",
    "is_rush_hour",
    "is_weekend",
    "is_late_night",
    "pickup_month",
]
fig = make_subplots(rows=2, cols=3, subplot_titles=cols_plot)
for i, c in enumerate(cols_plot, start=1):
    r, col = (i - 1) // 3 + 1, (i - 1) % 3 + 1
    fig.add_trace(go.Histogram(x=df_t[c], name=c, showlegend=False), row=r, col=col)
fig.update_layout(title_text="Distribution des Features Temporelles", height=650)
style_plotly(fig)
fig.show()


## Tip moyen par heure × jour de la semaine

Après `prepare_dataframe` pour obtenir `tip_pct`.

In [ ]:
df_p = prepare_dataframe(generate_taxi_df(80_000))
df_p["hour"] = df_p["tpep_pickup_datetime"].dt.hour
df_p["dow"] = df_p["tpep_pickup_datetime"].dt.dayofweek
heat = df_p.groupby(["dow", "hour"])["tip_pct"].mean().reset_index()

fig = px.density_heatmap(
    heat,
    x="hour",
    y="dow",
    z="tip_pct",
    title="Tip moyen par heure et jour de la semaine",
    labels={"dow": "Jour (0=lun)", "hour": "Heure", "tip_pct": "tip_pct moyen"},
    color_continuous_scale="Viridis",
)
style_plotly(fig)
fig.show()


## Smoothing bayésien — effet sur la variance

`compare_smoothing_effect` retourne un tableau ; courbe **variance vs m** (échelle log).

In [ ]:
df_feat = generate_taxi_df(60_000).copy()
df_feat["tip_pct"] = (df_feat["tip_amount"] / df_feat["fare_amount"].clip(lower=0.01)).clip(0, 3)

import io
import contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    smooth_df = compare_smoothing_effect(
        df_feat,
        "PULocationID",
        "tip_pct",
        [0.1, 1, 5, 10, 50, 100, 1000],
    )

fig = px.line(
    smooth_df,
    x="smoothing",
    y="variance",
    log_x=True,
    log_y=True,
    markers=True,
    title="Effet du Smoothing Bayésien sur la Variance",
)
style_plotly(fig)
fig.show()


## Bayesian vs Cross-Fitted target encoding

Distributions des encodages de `PULocationID` (même cible `tip_pct`).

In [ ]:
df_e = generate_taxi_df(40_000).copy()
df_e["tip_pct"] = (df_e["tip_amount"] / df_e["fare_amount"].clip(lower=0.01)).clip(0, 2)

be = BayesianTargetEncoder(columns=["PULocationID"], target="tip_pct", smoothing=10.0)
df_b = be.fit_transform(df_e.copy())
col_b = "PULocationID_target_enc"

ce = CrossFittedTargetEncoder(columns=["PULocationID"], target="tip_pct", smoothing=10.0, n_splits=5)
df_c = ce.fit_transform(df_e.copy())
col_c = "PULocationID_cross_enc"

fig = go.Figure()
fig.add_trace(go.Histogram(x=df_b[col_b], name="Bayesian", opacity=0.55, nbinsx=50))
fig.add_trace(go.Histogram(x=df_c[col_c], name="Cross-fitted", opacity=0.55, nbinsx=50))
fig.update_layout(barmode="overlay", title="Bayesian vs Cross-Fitted Target Encoding")
style_plotly(fig)
fig.show()

plt.figure(figsize=(8, 4))
sns.kdeplot(df_b[col_b], label="Bayesian", fill=True, alpha=0.3)
sns.kdeplot(df_c[col_c], label="Cross-fitted", fill=True, alpha=0.3)
plt.title("KDE des encodages")
plt.legend()
plt.tight_layout()
plt.show()


## Benchmark d’imputation (RMSE)

`ImputationBenchmark` sur colonnes numériques ; barres colorées du pire au meilleur (score composite RMSE moyen).

In [ ]:
df_imp = generate_taxi_df(25_000).copy()
bm = ImputationBenchmark(
    columns=["passenger_count", "trip_distance"],
    missing_pct=0.05,
    n_neighbors=5,
    n_mice_iter=3,
)
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    bm.run(df_imp)

summ = bm.summary()
scores = []
for _, row in summ.iterrows():
    rmses = [row[c] for c in summ.columns if c.startswith("rmse_") and pd.notna(row[c])]
    scores.append({"strategie": row["strategie"], "rmse_mean": float(np.mean(rmses)) if rmses else np.nan})

score_df = pd.DataFrame(scores).sort_values("rmse_mean", ascending=False)
worst, best = score_df["rmse_mean"].max(), score_df["rmse_mean"].min()
colors = [
    f"rgb({int(255 * (v - best) / (worst - best + 1e-9))}, {int(200 * (1 - (v - best) / (worst - best + 1e-9)))}, 80)"
    for v in score_df["rmse_mean"]
]

fig = go.Figure(go.Bar(x=score_df["strategie"], y=score_df["rmse_mean"], marker_color=colors))
fig.update_layout(title="Comparaison RMSE des Stratégies d'Imputation", yaxis_title="RMSE moyen (colonnes cibles)")
style_plotly(fig)
fig.show()


## Série temporelle — zone 132

`TimeSeriesFeaturizer` : série `trip_count` + moyenne mobile 24h (sans fuite, déjà intégrée dans le module).

In [ ]:
df_ts_in = generate_taxi_df(120_000)
featurizer = TimeSeriesFeaturizer(lags=[1, 24], windows=[3, 24], n_harmonics=3)
df_ts = featurizer.fit_transform(df_ts_in.copy())

z = df_ts[df_ts["PULocationID"] == 132].sort_values("pickup_hour_ts")
if z.empty:
    z = df_ts.sort_values("pickup_hour_ts").groupby("pickup_hour_ts").head(1)  # fallback

fig = go.Figure()
fig.add_trace(go.Scatter(x=z["pickup_hour_ts"], y=z["trip_count"], name="trip_count", mode="lines"))
if "trip_count_roll_mean_24h" in z.columns:
    fig.add_trace(
        go.Scatter(x=z["pickup_hour_ts"], y=z["trip_count_roll_mean_24h"], name="roll mean 24h", mode="lines")
    )
fig.update_layout(title="Activité zone 132 — Série temporelle")
style_plotly(fig)
fig.show()


## Harmoniques de Fourier (aperçu sur 48 heures)

Synthèse des `sin_k` / `cos_k` pour visualiser la forme d’onde journalière.

In [ ]:
t48 = pd.date_range("2023-06-01", periods=48, freq="h")
demo = pd.DataFrame({"pickup_hour_ts": t48, "PULocationID": 1, "trip_count": np.random.poisson(30, 48)})
demo = add_fourier_features(demo, n_harmonics=5)

fig = make_subplots(rows=5, cols=2, subplot_titles=[f"{t}{k}" for k in range(1, 6) for t in ("sin", "cos")])
for k in range(1, 6):
    fig.add_trace(go.Scatter(x=demo["pickup_hour_ts"], y=demo[f"fourier_sin_{k}"], showlegend=False), row=k, col=1)
    fig.add_trace(go.Scatter(x=demo["pickup_hour_ts"], y=demo[f"fourier_cos_{k}"], showlegend=False), row=k, col=2)
fig.update_layout(title="Harmoniques de Fourier (k=1 à 5)", height=900)
style_plotly(fig)
fig.show()
